In [1]:
!pip install transformers accelerate bitsandbytes peft gradio 
!pip install faiss-cpu sentence-transformers langchain-community langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 90.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 57.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 543.9/543.9 kB 38.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.15
    Uninstalling langchain-core-1.2.15:
      Successfully uninstalled langchain-core-1.2.15
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following depe

In [ ]:
import os
import re
import csv
import gc
import torch
import faiss
import pickle
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer
from sentence_transformers import SentenceTransformer

try:
    from transformers import BitsAndBytesConfig
except ImportError:
    BitsAndBytesConfig = None

try:
    from peft import PeftModel
except ImportError:
    PeftModel = None
    
def flush_memory():
    """Hàm dọn VRAM cứu rỗi Kaggle"""
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()

flush_memory()
# ====== Config Path ======
BASE_MODEL = "SeaLLMs/SeaLLM-7B-v2"
EMBEDDING_MODEL_NAME = "keepitreal/vietnamese-sbert"
#Đường dẫn file dữ liệu thực tế của bạn trên Kaggle
DATASET_ROOT = "/kaggle/input/datasets/nguyentuan205/fine-tuned-dataset/fine-tuned-model"
FAISS_INDEX_PATH = os.path.join(DATASET_ROOT, "faiss_index", "index.faiss")
FAISS_PKL_PATH   = os.path.join(DATASET_ROOT, "faiss_index", "index.pkl")
FINE_TUNED_CHECKPOINT_ROOT = os.path.join(DATASET_ROOT, "finetuned_model")

# ====== Global State ======
base_model_obj = None
tokenizer = None
active_model = None
current_checkpoint = None

# RAG Global State
vector_db = None
corpus_metadata = None
embed_model = None

# ====== RAG Functions ======
def load_rag_assets():
    global vector_db, corpus_metadata, embed_model
    if vector_db is not None:
        return True
    
    if not FAISS_INDEX_PATH or not os.path.exists(FAISS_INDEX_PATH):
        print("⚠️ File FAISS không tồn tại.")
        return False

    try:
        print("--- Loading RAG Assets ---")
        vector_db = faiss.read_index(FAISS_INDEX_PATH)
        with open(FAISS_PKL_PATH, "rb") as f:
            corpus_metadata = pickle.load(f)
        embed_model = SentenceTransformer(EMBEDDING_MODEL_NAME)
        print("✅ RAG Assets Loaded.")
        return True
    except Exception as e:
        print(f"❌ Lỗi load RAG: {e}")
        return False

def retrieve_rag_context(query, top_k=3):
    if not load_rag_assets():
        return []
    
    query_vector = embed_model.encode([query])
    distances, indices = vector_db.search(query_vector, top_k)
    
    results = []
    for idx in indices[0]:
        if idx != -1 and idx < len(corpus_metadata):
            content = corpus_metadata[idx]
            # Xử lý nếu metadata là dict hoặc string
            text = content if isinstance(content, str) else content.get('text', str(content))
            results.append({"text": text})
    return results

# ====== Model Loading Logic ======
def _quantization_args():
    if BitsAndBytesConfig is None: return None
    return BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

def _ensure_base_loaded(progress):
    global base_model_obj, tokenizer
    if base_model_obj is not None: return None

    progress(0.1, desc="Load tokenizer...")
    quant_config = _quantization_args()
    
    try:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
        progress(0.4, desc="Load base model 4-bit...")
        base_model_obj = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL,
            quantization_config=quant_config,
            device_map="auto",
            trust_remote_code=True,
        )
        base_model_obj.eval()
    except Exception as e:
        return f"⚠️ Lỗi: {e}"
    return None

def load_model(model_source, use_rag, checkpoint_name, progress=gr.Progress()):
    global active_model, current_checkpoint
    
    err = _ensure_base_loaded(progress)
    if err: return err

    if model_source == "Base Model":
        active_model = base_model_obj
        current_checkpoint = None
        return "✅ Đã chuyển sang Base Model."

    # Load Fine-tuned
    if not checkpoint_name: return "⚠️ Chọn checkpoint!"
    if current_checkpoint == checkpoint_name: return f"✅ Đã sẵn sàng [{checkpoint_name}]"

    progress(0.7, desc=f"Gắn adapter {checkpoint_name}...")
    adapter_path = os.path.join(FINE_TUNED_CHECKPOINT_ROOT, checkpoint_name)
    try:
        active_model = PeftModel.from_pretrained(
            base_model_obj, adapter_path, torch_dtype=torch.float16, device_map="auto"
        )
        active_model.eval()
        current_checkpoint = checkpoint_name
        return f"✅ Đã load adapter: {checkpoint_name}"
    except Exception as e:
        return f"⚠️ Lỗi gắn adapter: {e}"

# ====== Prompt & Generate ======
def build_prompt(history, system_prompt, rag_context=None):
    messages = []
    if rag_context:
        messages.append({"role": "system", "content": f"Sử dụng thông tin sau để trả lời:\n{rag_context}"})
    if system_prompt.strip():
        messages.append({"role": "system", "content": system_prompt.strip()})

    for user_msg, assistant_msg in history:
        messages.append({"role": "user", "content": user_msg})
        if assistant_msg is not None:  # Thêm check này
            messages.append({"role": "assistant", "content": assistant_msg})

    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def generate(history, user_message, system_prompt, max_new_tokens, temperature, top_p,
             model_source, use_rag, checkpoint_name):
    if not user_message.strip(): return history, ""
    if active_model is None: return history + [[user_message, "⚠️ Hãy nhấn 'Load Model'"]], ""

    rag_context = None
    if use_rag:
        docs = retrieve_rag_context(user_message)
        if docs:
            rag_context = "\n\n---\n\n".join([d["text"] for d in docs])

    history = history + [[user_message, None]]
    prompt = build_prompt(history, system_prompt, rag_context)
    inputs = tokenizer(prompt, return_tensors="pt").to(active_model.device)

    with torch.no_grad():
        output_ids = active_model.generate(
            **inputs,
            max_new_tokens=int(max_new_tokens),
            temperature=float(temperature) if temperature > 0 else 1.0,
            top_p=float(top_p),
            do_sample=temperature > 0,
            repetition_penalty=1.1,
            eos_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True).strip()
    history[-1][1] = response
    return history, ""

# ====== Helpers ======
def get_checkpoints():
    if not FINE_TUNED_CHECKPOINT_ROOT or not os.path.isdir(FINE_TUNED_CHECKPOINT_ROOT): return []
    return sorted([d for d in os.listdir(FINE_TUNED_CHECKPOINT_ROOT) if d.startswith("checkpoint-")],
                  key=lambda x: int(x.split("-")[-1]))

# ====== UI ======
checkpoints = get_checkpoints()
with gr.Blocks(title="SeaLLM RAG FAISS", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 🌊 SeaLLM-7B-v2 + FAISS RAG")
    
    with gr.Row():
        with gr.Column(scale=1):
            model_choice = gr.Dropdown(choices=["Base Model", "Fine-tuned Model"], value="Fine-tuned Model", label="Model")
            checkpoint_choice = gr.Dropdown(choices=checkpoints, value=checkpoints[-1] if checkpoints else None, label="Checkpoint")
            load_btn = gr.Button("🔄 Load Model", variant="primary")
            load_status = gr.Textbox(label="Status", interactive=False)
            rag_checkbox = gr.Checkbox(label="🔍 Bật FAISS RAG", value=True)
            
            with gr.Accordion("Parameters", open=False):
                system_p = gr.Textbox(value="You are a helpful assistant.", label="System Prompt")
                max_t = gr.Slider(64, 2048, 512, step=64, label="Max Tokens")
                temp = gr.Slider(0.0, 1.0, 0.3, label="Temperature")
                p_val = gr.Slider(0.1, 1.0, 0.9, label="Top-p")

        with gr.Column(scale=3):
            chatbot = gr.Chatbot(height=550)
            msg = gr.Textbox(placeholder="Nhập câu hỏi tại đây...", container=False, scale=4)
            with gr.Row():
                send = gr.Button("Gửi", variant="primary")
                clear = gr.Button("Xóa lịch sử")

    # Events
    load_btn.click(load_model, [model_choice, rag_checkbox, checkpoint_choice], [load_status])
    send.click(generate, [chatbot, msg, system_p, max_t, temp, p_val, model_choice, rag_checkbox, checkpoint_choice], [chatbot, msg])
    msg.submit(generate, [chatbot, msg, system_p, max_t, temp, p_val, model_choice, rag_checkbox, checkpoint_choice], [chatbot, msg])
    clear.click(lambda: [], None, [chatbot])

demo.launch(share=True)

/tmp/ipykernel_57/1582092808.py:197: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="SeaLLM RAG FAISS", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_57/1582092808.py:215: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(height=550)
/tmp/ipykernel_57/1582092808.py:215: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(height=550)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8a2547128413968b30.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


config.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

bpe.codes: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ RAG Assets Loaded.


Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.
